# Manejo de datos espaciales en Python: de tablas a geometrías

**Curso:** Análisis Geoespacial, Universidad Nacional de Colombia

En este notebook se construyen datos tabulares con `pandas` antes de incorporar su componente geográfica. Esta separación es importante: una tabla contiene atributos, mientras que un `GeoDataFrame` agrega una columna de geometrías y un sistema de referencia de coordenadas.

## Objetivos

Al finalizar, se podrá:

- Crear un `DataFrame` escribiendo los datos directamente en Python.
- Importar tablas desde archivos CSV y Excel.
- Convertir diccionarios, tuplas y matrices en tablas.
- Reconocer qué información hace falta para convertir una tabla en datos espaciales.

## 1. Preparar el entorno

`pandas` es la biblioteca principal para trabajar con tablas. Por convención se importa con el alias `pd`.

In [ ]:
from pathlib import Path

import pandas as pd

pd.__version__

## 2. Crear una tabla directamente en Python

Un `DataFrame` es una estructura tabular con filas y columnas. Cada fila representa una observación y cada columna una variable. Para comenzar, se puede construir a partir de un diccionario cuyas claves serán los nombres de las columnas.

In [ ]:
datos_municipios = {
    "municipio": ["Medellín", "Bello", "Envigado", "Itagüí"],
    "poblacion_miles": [2569, 552, 250, 291],
    "area_km2": [376, 149, 79, 21]
}

municipios = pd.DataFrame(datos_municipios)
municipios

Cada columna de un `DataFrame` tiene un tipo de dato. Es conveniente revisarlo antes de hacer cálculos o mapas.

In [ ]:
municipios.info()

municipios["densidad_hab_km2"] = (
    municipios["poblacion_miles"] * 1000 / municipios["area_km2"]
)
municipios.round({"densidad_hab_km2": 1})

## 3. Crear una tabla con filas escritas directamente

Otra forma clara es escribir una lista de diccionarios. Cada diccionario es una fila y sus claves son las columnas. Este formato es útil cuando las observaciones se van agregando una por una.

In [ ]:
registros = [
    {"id": 1, "sitio": "Quebrada La Iguaná", "lluvia_mm": 42.5},
    {"id": 2, "sitio": "Quebrada Santa Elena", "lluvia_mm": 35.2},
    {"id": 3, "sitio": "Quebrada La Picacha", "lluvia_mm": 51.8}
]

lluvias = pd.DataFrame(registros)
lluvias

## 4. Importar un archivo CSV

Un archivo CSV almacena valores separados por un delimitador, normalmente coma o punto y coma. La función `read_csv()` importa el archivo y devuelve un `DataFrame`.

El siguiente ejemplo crea un CSV pequeño para que el notebook sea reproducible. En un proyecto real, se reemplaza la ruta por la ubicación del archivo entregado por la fuente de datos.

In [ ]:
carpeta_trabajo = Path.cwd()
archivo_csv = carpeta_trabajo / "ejemplo_estaciones.csv"

lluvias.to_csv(archivo_csv, index=False, encoding="utf-8")
datos_csv = pd.read_csv(archivo_csv)
datos_csv

Si el archivo usa punto y coma como separador, se debe indicarlo explícitamente:

```python
datos = pd.read_csv("datos.csv", sep=";", encoding="utf-8")
```

Después de importar, conviene comprobar dimensiones, nombres, tipos y valores faltantes.

In [ ]:
datos_csv.shape, datos_csv.columns.tolist(), datos_csv.isna().sum()

## 5. Importar un archivo de Excel

La función `read_excel()` permite leer una hoja de cálculo. Para archivos `.xlsx`, `pandas` utiliza normalmente el motor `openpyxl`. El siguiente ejemplo genera un archivo temporal y luego lo importa.

In [ ]:
archivo_excel = carpeta_trabajo / "ejemplo_estaciones.xlsx"
lluvias.to_excel(archivo_excel, sheet_name="estaciones", index=False)

datos_excel = pd.read_excel(archivo_excel, sheet_name="estaciones")
datos_excel

Para leer otra hoja se cambia `sheet_name`. Si se desea inspeccionar los nombres de las hojas:

```python
libro = pd.ExcelFile("datos.xlsx")
libro.sheet_names
```

## 6. Caso de un diccionario

Un diccionario relaciona claves con valores. Si cada clave representa una variable y cada valor es una lista de igual longitud, se convierte directamente en un `DataFrame`.

In [ ]:
coordenadas = {
    "lugar": ["Medellín", "Bello", "Envigado"],
    "longitud": [-75.5636, -75.5578, -75.5890],
    "latitud": [6.2518, 6.3373, 6.1706]
}

tabla_coordenadas = pd.DataFrame(coordenadas)
tabla_coordenadas

En este caso las coordenadas todavía son columnas numéricas. La tabla no es espacial hasta crear geometrías y definir el sistema de referencia de coordenadas.

## 7. Caso de una tupla

Una tupla es una secuencia ordenada e inmutable. Una lista de tuplas es apropiada cuando cada tupla representa una fila y todas las filas tienen el mismo orden de variables. En ese caso, se deben proporcionar los nombres de las columnas.

In [ ]:
filas = [
    ("Medellín", 6.2518, -75.5636, 1495),
    ("Bello", 6.3373, -75.5578, 1450),
    ("Envigado", 6.1706, -75.5890, 1575)
]

columnas = ["municipio", "latitud", "longitud", "elevacion_m"]
tabla_tuplas = pd.DataFrame(filas, columns=columnas)
tabla_tuplas

## 8. Caso de una matriz

Una matriz puede representarse como una lista de listas o como un arreglo de `numpy`. Cada fila debe contener el mismo número de valores. Al convertirla en un `DataFrame`, se asignan los nombres de las columnas.

In [ ]:
import numpy as np

matriz = np.array([
    [1, 101.2, 6.2518, -75.5636],
    [2, 98.7, 6.3373, -75.5578],
    [3, 110.4, 6.1706, -75.5890]
])

tabla_matriz = pd.DataFrame(
    matriz,
    columns=["id", "lluvia_mm", "latitud", "longitud"]
)
tabla_matriz

Al crear una matriz numérica, `numpy` puede almacenar todos los valores con un tipo común. Por eso se debe revisar el tipo de las columnas y convertirlas cuando sea necesario.

In [ ]:
tabla_matriz.dtypes

tabla_matriz["id"] = tabla_matriz["id"].astype(int)
tabla_matriz

## 9. De una tabla a un `GeoDataFrame`

Cuando una tabla incluye longitud y latitud, se puede crear una geometría de puntos con `geopandas`. El sistema EPSG:4326 corresponde a coordenadas geográficas en longitud y latitud.

Esta celda requiere que `geopandas` esté instalado en el entorno del curso.

In [ ]:
import geopandas as gpd

puntos = gpd.GeoDataFrame(
    tabla_coordenadas.copy(),
    geometry=gpd.points_from_xy(
        tabla_coordenadas["longitud"],
        tabla_coordenadas["latitud"]
    ),
    crs="EPSG:4326"
)

puntos

## Geometrías y operaciones espaciales

Antes de convertir una tabla en un `GeoDataFrame`, conviene conocer los objetos geométricos básicos. La biblioteca `Shapely` permite crear puntos, líneas y polígonos, y consultar sus propiedades espaciales.

In [ ]:
from shapely.geometry import LineString, Point, Polygon

# Crear un punto a partir de una coordenada x, y
punto_1 = Point(-75.5636, 6.2518)
punto_2 = Point(-75.5578, 6.3373)

# Crear una línea a partir de una secuencia de coordenadas
linea = LineString([
    (-75.5636, 6.2518),
    (-75.5578, 6.3373),
    (-75.5890, 6.1706)
])

# Crear un polígono: el primer y último punto deben coincidir
poligono = Polygon([
    (-75.60, 6.20),
    (-75.54, 6.20),
    (-75.54, 6.30),
    (-75.60, 6.30),
    (-75.60, 6.20)
])

print(punto_1)
print(linea)
print(poligono)

In [ ]:
# Propiedades del punto
print("Tipo:", punto_1.geom_type)
print("Coordenada x:", punto_1.x)
print("Coordenada y:", punto_1.y)
print("Coordenadas:", tuple(punto_1.coords))
print("Límites:", punto_1.bounds)

# Propiedades de la línea
print("\nLongitud de la línea:", linea.length)
print("Centroide de la línea:", linea.centroid)
print("Coordenadas de la línea:", list(linea.coords))

# Propiedades del polígono
print("\nÁrea del polígono:", poligono.area)
print("Perímetro del polígono:", poligono.length)
print("Centroide del polígono:", poligono.centroid)
print("Límites del polígono:", poligono.bounds)

# Operaciones entre geometrías
print("\nDistancia entre puntos:", punto_1.distance(punto_2))
print("¿El polígono contiene el punto 1?:", poligono.contains(punto_1))
print("¿La línea cruza el polígono?:", linea.intersects(poligono))

In [ ]:
# Una geometría no almacena atributos temáticos por sí sola.
# Los atributos se organizan en una tabla con una fila por geometría.
atributos = pd.DataFrame({
    "id": [1, 2],
    "sitio": ["Medellín", "Bello"],
    "temperatura_c": [23.4, 22.8]
})

puntos_atributos = gpd.GeoDataFrame(
    atributos,
    geometry=[punto_1, punto_2]
)

puntos_atributos

In [ ]:
# Al crear el GeoDataFrame desde cero, el CRS todavía no está definido.
print("CRS inicial:", puntos_atributos.crs)

# EPSG:4326 corresponde a WGS84, con coordenadas de longitud y latitud.
puntos_atributos = puntos_atributos.set_crs("EPSG:4326")
print("CRS asignado:", puntos_atributos.crs)

# set_crs() asigna el sistema conocido; no cambia los valores de las coordenadas.
# to_crs() transforma las coordenadas a otro sistema.
puntos_proyectados = puntos_atributos.to_crs("EPSG:3116")
puntos_proyectados

Ejemplo

In [ ]:
df = pd.DataFrame(
    {'City': ['Buenos Aires', 'Brasilia', 'Santiago', 'Bogota', 'Caracas'],
     'Country': ['Argentina', 'Brazil', 'Chile', 'Colombia', 'Venezuela'],
     'Latitude': [-34.58, -15.78, -33.45, 4.60, 10.48],
     'Longitude': [-58.66, -47.91, -70.66, -74.08, -66.86]})
df

In [ ]:
df.plot()

In [ ]:
df['Coordinates']  = list(zip(df['Longitude'], df.Latitude))
df

In [ ]:
df['Coordinates'] = df['Coordinates'].apply(Point)
df

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry='Coordinates')
gdf

In [ ]:
gdf.plot()

Ejemplo

In [ ]:
# Create an empty geopandas GeoDataFrame
newdata = gpd.GeoDataFrame()
# Let's see what's inside
newdata

In [ ]:
# Coordinates of the Helsinki Senate square in Decimal Degrees
coordinates = [(24.950899, 60.169158), (24.953492, 60.169158), (24.953510, 60.170104), (24.950958, 60.169990)]

# Create a Shapely polygon from the coordinate-tuple list
poly = Polygon(coordinates)

# Let's see what we have
poly

In [ ]:
# Insert the polygon into 'geometry' -column at index 0
newdata.loc[0, 'geometry'] = poly

# Let's see what we have now
newdata

In [ ]:
# Add a new column and insert data
newdata.loc[0, 'Location'] = 'Senaatintori'

# Let's check the data
newdata

In [ ]:
print(newdata.crs)

In [ ]:
from fiona.crs import from_epsg
newdata.crs = from_epsg(4326)
newdata.crs

## 10. Crear un `GeoDataFrame` desde un CSV

Un CSV con columnas `longitude` y `latitude` todavía es un `DataFrame` convencional. Primero se importa la tabla; después se convierten las coordenadas en geometrías de puntos.

In [ ]:
tokyo_url = "https://geographicdata.science/book/_downloads/7fb86b605af15b3c9cbd9bfcbead23e9/tokyo_clean.csv"

tokyo_df = pd.read_csv(tokyo_url)
tokyo_df.head()

In [ ]:
tokyo_gdf = gpd.GeoDataFrame(
    tokyo_df,
    geometry=gpd.points_from_xy(
        x=tokyo_df["longitude"],
        y=tokyo_df["latitude"],
        crs="EPSG:4326"
    )
)

tokyo_gdf.head()
print("Tipo:", type(tokyo_gdf))
print("CRS:", tokyo_gdf.crs)

## 11. Importar directamente un archivo espacial

Un shapefile ya contiene geometrías y atributos. Por eso `gpd.read_file()` lo carga directamente como un `GeoDataFrame`; no es necesario crear la columna `geometry` manualmente.

In [ ]:
countries = gpd.read_file(
    "MyDrive/CATEDRA/ANALISISGEOESPACIAL/AnalisisGeoespacial/data/Notebooks/mundo/ne_10m_admin_0_map_units.shp"
)

print("Tipo:", type(countries))
print("CRS:", countries.crs)
countries.head()

In [ ]:
puntos.plot()

Exportar

In [ ]:
uk.to_file("data/uk.shp", crs={'init' :'epsg:4326'})

In [ ]:
colombia.to_file("data/colombia.geojson", driver="GeoJSON")